# Train/test EC classification

Train a graph classifier with a homology-aware train/test split. Notebook 03 fetches canonical UniProt sequences and uses MMseqs2 to cluster proteins at the configured identity and coverage thresholds. Entire sequence clusters stay on one side of the split, preventing close homologues from leaking across train and test. Classes represented by only one homology cluster are retained in training and reported as train-only. Protein graph vectorization shows a progress bar during training and prediction. XGBoost trains directly on sparse features with parallel histogram trees, with a progress bar for the 300 boosting rounds.

In [7]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from enzyme_workflow.classifier import run_classifier


In [8]:
# User parameters
TEST_SIZE = 0.20
RANDOM_STATE = 42
HOMOLOGY_MIN_SEQUENCE_IDENTITY = 0.30
HOMOLOGY_MIN_COVERAGE = 0.80  # coverage of the shorter sequence
NODE_LABEL_ALPHABET = "aa20"  # aa20, hp2, charge4, physchem5, chem7, or dayhoff6
BASE_ESTIMATOR_KIND = "linear_sgd"  # xgboost, linear_sgd, or random_forest
SIMPLEX_DIMENSION = (0,3)  # inclusive: integer means 0..max; tuple means (min, max)
CLIQUE_ENUMERATOR = "auto"  # auto: fast for dimensions 0–3; generic above 3; or force fast/generic
PAIR_DISTANCE = (1,1)  # inclusive: integer means 0..max; tuple means (min, max)
NBITS = 11
VECTORIZER_PROGRESS = True
CLASSIFIER_PROGRESS = True
N_JOBS = -1  # use all available CPU cores for vectorization and classifier fitting


In [9]:
classification = run_classifier(
    PROJECT_ROOT,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    homology_min_sequence_identity=HOMOLOGY_MIN_SEQUENCE_IDENTITY,
    homology_min_coverage=HOMOLOGY_MIN_COVERAGE,
    node_label_alphabet=NODE_LABEL_ALPHABET,
    base_estimator_kind=BASE_ESTIMATOR_KIND,
    simplex_dimension=SIMPLEX_DIMENSION,
    clique_enumerator=CLIQUE_ENUMERATOR,
    pair_distance=PAIR_DISTANCE,
    nbits=NBITS,
    vectorizer_progress=VECTORIZER_PROGRESS,
    classifier_progress=CLASSIFIER_PROGRESS,
    n_jobs=N_JOBS,
)
print(f"Split: {classification.split_strategy}")
print(f"Accuracy: {classification.accuracy:.3f}; balanced accuracy: {classification.balanced_accuracy:.3f}")
print(f"Saved run: {classification.run_dir}")
display(classification.class_counts)
display(classification.class_group_counts)
if classification.train_only_classes:
    print("Train-only EC classes (single UniProt group):", classification.train_only_classes)
display(classification.train_counts)
display(classification.test_counts)
display(classification.report)
display(classification.confusion)
display(classification.confusion_figure)
plt.close(classification.confusion_figure)
display(classification.confusion_fractions)
display(classification.confusion_fraction_figure)
plt.close(classification.confusion_fraction_figure)
display(classification.roc_figure)
plt.close(classification.roc_figure)
display(classification.pr_figure)
plt.close(classification.pr_figure)
display(classification.predictions.head())


Homology grouping: MMseqs2, >=30% sequence identity over >=80% of the shorter sequence
Fitting classifier on 10,702 training proteins


Vectorizing protein graphs:   0%|          | 0/10702 [00:00<?, ?protein/s]

KeyboardInterrupt: 

ROC and PR curves use micro-averaged one-vs-rest scores. The ROC dashed diagonal shows chance-level discrimination; the PR dashed horizontal line shows the positive-label prevalence baseline. Confusion fractions are normalized by true-class row, so each nonempty row sums to one. All four figures use square plotting axes.
